# Prepare image request GeoJSON

#### A subset that contains SME request parameters for image acquisaitions at sites from a vendor using the CSDA Evaluation Sites GeoJSON

Paul Montesano, PhD  
June 2026

Builds the **site request GeoJSON** that accompanies a data order to a vendor.

### What problem this solves

Our authoritative site inventory lives in a GeoJSON (derived from the Evaluation Sites
database). For any given vendor request, a collaborator fills in a Google Sheet
specifying *which* sites to request and *what acquisition parameters* apply
(cloud limits, view angle, number of acquisitions, products, etc.).

This notebook merges the two, and exports only the requested sites with their
final parameters.

**The governing rule:** the sheet is a **sparse patch**, not a full record.

- A cell with a value → **overwrites** the GeoJSON
- A **blank** cell → **leaves the GeoJSON value untouched**
- A cell containing `0` → is a real zero, honoured as such

This distinction matters. An earlier version of this code represented "no value in
the sheet" as the number `0`, which silently overwrote good GeoJSON values with
zeros. Blank now means blank.

---

## Inputs

| Input | Source | Role |
|---|---|---|
| `sites` | standard sites GeoJSON | Authoritative geometries + baseline attributes. Read-only. |
| Eval site request sheet | Google Sheet with `Site Name` and relevant cols copied from Eval Sites dB, one tab per request | Which sites, and which parameters to change. |
| `VENDOR_SHEETS` | dict in the notebook/lib | Maps a vendor name → `(sheet_id, gid)` |

The sheet must be **link-shared** (anyone with the link can view) — it is read via
Google's CSV export endpoint. If it isn't shared, `pd.read_csv` receives an HTML
login page and fails with a parser error.

## Output

`csda_sites_aoi_<VENDOR_NAME>.geojson` — one feature per requested site, carrying
the merged acquisition parameters.

---

## Field ownership

Some fields come from the sheet; some are owned by the GeoJSON and **cannot** be
patched. This is deliberate.

| Field | Owner | Notes |
|---|---|---|
| `geometry` | GeoJSON | Built from the Evaluation Sites DB |
| `area_km2` | **Computed** | Recalculated geodesically from `geometry` on every run |
| `aoi_shape`, `aoi_size_km` | Sheet (patchable) | `aoi_size_km` is **dropped on export** |
| `max_aoi_cloud_pct`, `max_scene_cloud_pct` | Sheet | |
| `max_view_angle` | Sheet | |
| `min_num_acqs`, `ideal_num_acqs` | Sheet | |
| `products`, `country`, `remote_sensing_domain`, `evaluation_category` | Sheet | |

Two points worth understanding:

- **`area_km2` is never read from the sheet.** It is derived from the actual
  polygon using the WGS84 ellipsoid, so it is always consistent with the geometry
  being sent. (A 3 km-radius circle yields ≈ 28.23 km², not exactly `πr²` = 28.27,
  because the calculation is ellipsoidal.)
- **`aoi_size_km` is dropped from the export** because it confuses recipients —
  it's ambiguous whether it means a radius, a diameter, or a box edge. `area_km2`
  states the AOI size unambiguously.

---

## Workflow

```python
VENDOR_NAME = "HydroSat_Kerry"
sheet_id, gid = VENDOR_SHEETS[VENDOR_NAME]
sheet_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/edit?usp=sharing"

# 1. Read the sheet into a sparse params table (NaN = "no opinion")
params = load_site_params(sheet_url, gid=gid)
display(params)

# 2. Patch the FULL sites frame — never a column subset
sites_updated, changes = apply_site_params(sites, params)

# 3. Review what actually changed
display(changes.query("change != 'unchanged'"))

# 4. Subset + export
#    (see the export cell for column selection)
```

---

## Reading the diagnostics

### `load_site_params` prints a **cell fill rate**

```
cell fill rate:
    aoi_shape            100 %
    max_view_angle       100 %
    products               0 %   ← ✗ investigate
```

**A parameter at `0 %` almost always means the column alias didn't match**, not
that the sheet is empty. Sheet headers are normalized by lowercasing and stripping
all non-alphanumeric characters (`"Max AOI Cloud %"` → `"maxaoicloud"`), then
matched **exactly** against `PARAM_ALIASES`. Check the normalized form of your
header and add it to the alias list if needed.

It also reports:
- `⚠️ no column in sheet for: [...]` — declared parameters with no matching column
- `ℹ️ sheet columns ignored: [...]` — sheet columns nothing maps to (often fine;
  a typo'd header shows up here)

### `apply_site_params` returns a change report

Every cell is classified:

| `change` | Meaning |
|---|---|
| `unchanged` | Sheet agreed with the GeoJSON, or said nothing |
| `filled` | GeoJSON was empty; sheet supplied a value |
| `overwritten` | **GeoJSON had a different value; the sheet replaced it** |

**Always read the `overwritten` block.** Sometimes intended, sometimes a
sheet-entry error — either way it is now visible rather than silent.

Also check the match count:

```
12 of 47 GeoJSON sites matched the sheet
  ⚠️  in sheet but not in GeoJSON: ['lake tahoe west']
```

Anything in that warning is a **site you asked for that will not be exported**.
Site names are matched case- and punctuation-insensitively, but a genuinely
different name won't match. Fix the sheet or the GeoJSON before ordering.

---

## Common problems

| Symptom | Cause |
|---|---|
| `ParserError` reading the sheet | Sheet not link-shared, or wrong `gid` |
| A parameter shows `0 %` fill | Header alias mismatch — see above |
| Fewer sites exported than requested | Site-name mismatch; see the match warning |
| Unexpected values in `overwritten` | Sheet cell contains something you didn't intend |
| `area_km2` disagrees with `aoi_size_km` | Geometry wasn't rebuilt after the shape/size changed |

### The AOI consistency check

If the sheet changes `aoi_shape` or `aoi_size_km`, the stored `geometry` no longer
matches those parameters — `area_km2` (from geometry) will disagree with what the
new shape/size imply. The notebook flags this:

```python
display(chk.query("mismatch_pct > 5"))
```

**Any site flagged here needs its AOI rebuilt from the Evaluation Sites DB**, not
just relabelled. This notebook does not rebuild geometries.

---

## Extending

**New vendor:** add an entry to `VENDOR_SHEETS` as `name: (sheet_id, gid)`. The
`gid` is the `gid=` value in the sheet URL and identifies the specific *tab*.

**New parameter:** add it to `PARAM_ALIASES` with every header spelling you expect
(normalized form: lowercase, alphanumeric only). If it's numeric, also add it to
`NUMERIC_PARAMS` so blanks coerce to `NaN` instead of strings.

**New derived field:** add it to `DERIVED_PARAMS` so the sheet can never patch it.

---

## Notes

- Numeric parameters are kept as **floats**, never cast to `int` — so `0.5` stays
  `0.5` and `30.0` is not reported as differing from `30`.
- `sites` itself is never mutated; `apply_site_params` works on a copy.
- Re-running is safe and idempotent.

In [3]:
import pandas as pd
import re
import geopandas as gpd
from datetime import datetime

In [4]:
import numpy as np
from pyproj import Geod

### Functions

In [5]:
# ── Canonical parameter -> accepted sheet headers (EXACT match after normalization) ──
# Normalization strips everything non-alphanumeric and lowercases, so list aliases
# the same way: 'Max AOI Cloud %' -> 'maxaoicloud'
PARAM_ALIASES = {
    "country":             ["country", "nation"],
    "remote_sensing_domain": ["remotesensingdomain", "domain", "rsdomain"],
    "evaluation_category": ["evaluationcategory", "evalcategory", "category"],
    "aoi_shape":           ["aoishape", "shape"],
    "aoi_size_km":         ["aoisizekm", "aoisize", "sizekm"],
    "max_cloud_pct_aoi":   ["maxcloudpctaoi", "maxcloud", "aoicloud"],
    "max_cloud_pct_scene": ["maxcloudpctscene", "maxcloud", "scenecloud"],
    "max_view_angle":      ["maxviewangle", "viewangle", "pointingangle"],
    "min_num_acqs":        ["minnumacqs", "minacqs", "minimumacqs"],
    "ideal_num_acqs":      ["idealnumacqs", "idealacqs"],
    "products":            ["products", "producttype", "productlevel"],
}
NUMERIC_PARAMS = {"aoi_size_km", "max_cloud_pct_aoi", "max_cloud_pct_scene",
                  "max_view_angle", "min_num_acqs", "ideal_num_acqs"}
SITE_ALIASES = ["sitename", "site"]

# area_km2 is GeoJSON-owned (derived from geometry) and deliberately NOT patchable.
DERIVED_PARAMS = ["area_km2"]


def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def load_site_params(sheet_url, gid=None, verbose=True):
    """Read a vendor sheet into a sparse params table indexed by site name.

    Empty cells stay NaN and will NOT overwrite anything downstream.
    """
    csv_url = re.sub(r"/edit.*", "/export?format=csv", sheet_url)
    if gid is not None:
        csv_url += f"&gid={gid}"
    raw = pd.read_csv(csv_url)

    headers = {_norm(c): c for c in raw.columns}

    site_col = next((headers[a] for a in SITE_ALIASES if a in headers), None)
    if site_col is None:
        raise ValueError(f"No site-name column found. Headers: {list(raw.columns)}")

    resolved, unresolved = {}, []
    for param, aliases in PARAM_ALIASES.items():
        hit = next((headers[a] for a in aliases if a in headers), None)
        if hit:
            resolved[param] = hit
        else:
            unresolved.append(param)

    out = pd.DataFrame({p: raw[col] for p, col in resolved.items()})
    out.insert(0, "site_name", raw[site_col].astype(str).str.strip())
    out = out[out["site_name"].notna() & (out["site_name"] != "") &
              (out["site_name"].str.lower() != "nan")]

    # blank/whitespace-only -> NaN ; numerics coerced (never silently 0, never int)
    for p in resolved:
        if p in NUMERIC_PARAMS:
            out[p] = pd.to_numeric(
                out[p].astype(str).str.replace(r"[^\d.\-]", "", regex=True)
                      .replace("", np.nan), errors="coerce")
        else:
            out[p] = out[p].astype(str).str.strip().replace(
                {"": np.nan, "nan": np.nan, "None": np.nan})

    out = out.set_index("site_name")
    if verbose:
            print(f"Sheet: {len(out)} sites, {len(resolved)} parameters resolved")
            used = set(resolved.values()) | {site_col}
            if unresolved:
                print(f"  ⚠️  no column in sheet for: {unresolved}")
            ignored = [c for c in raw.columns if c not in used]
            if ignored:
                print(f"  ℹ️  sheet columns ignored: {ignored}")
            if len(out):
                fill = (out.notna().mean() * 100).round(0).astype(int).astype(str) + " %"
                print("  cell fill rate:")
                print(fill.to_string().replace("\n", "\n    "))
    return out


def geodesic_area_km2(gdf):
    """Authoritative area straight from the geometry (WGS84 ellipsoid)."""
    geod = Geod(ellps="WGS84")

    def _area(geom):
        if geom is None or geom.is_empty:
            return np.nan
        parts = getattr(geom, "geoms", [geom])
        return sum(abs(geod.geometry_area_perimeter(p)[0]) for p in parts) / 1e6

    return gdf.to_crs(4326).geometry.map(_area)


def apply_site_params(sites_gdf, params_df, site_key="Site Name"):
    """Patch sites_gdf with non-null sheet values only. Returns (updated_gdf, report)."""
    before = sites_gdf.copy()

    # case/whitespace-insensitive join so name drift doesn't fail silently
    key = before[site_key].astype(str).str.strip()
    lut = {_norm(s): s for s in params_df.index}
    matched = key.map(lambda s: lut.get(_norm(s)))

    patch = (params_df.reindex(matched.values)
             .drop(columns=DERIVED_PARAMS, errors="ignore")
             .set_index(before.index))

    after = before.copy()
    for c in patch.columns:                      # add genuinely new params as NaN
        if c not in after.columns:
            after[c] = np.nan
    after.update(patch)                          # NaN cells in patch are skipped

    # area_km2 always recomputed from geometry, never taken from the sheet
    after["area_km2"] = geodesic_area_km2(after)

    report = _change_report(before, after, patch.columns, key, matched,
                            sheet_index=params_df.index)
    return after, report


def _change_report(before, after, param_cols, key, matched, sheet_index=None):
    rows = []
    for i in before.index:
        for c in param_cols:
            old = before[c].iloc[before.index.get_loc(i)] if c in before else np.nan
            new = after.loc[i, c]
            if pd.isna(old) and pd.isna(new):
                continue
            same = (pd.notna(old) and pd.notna(new) and (
                np.isclose(old, new, rtol=1e-6, equal_nan=True)
                if isinstance(new, (int, float, np.number)) and
                   isinstance(old, (int, float, np.number))
                else str(old).strip() == str(new).strip()))
            rows.append({
                "site": key.loc[i], "parameter": c, "old": old, "new": new,
                "change": "unchanged" if same
                          else "filled" if pd.isna(old)
                          else "overwritten",
            })
    rep = pd.DataFrame(rows)

    print("\n" + "=" * 62)
    print("\n" + "=" * 62)
    print(f"{matched.notna().sum()} of {len(before)} GeoJSON sites matched the sheet")
    if sheet_index is not None:
        unmatched = sorted(set(map(_norm, sheet_index)) - set(map(_norm, key)))
        if unmatched:
            print(f"  ⚠️  in sheet but not in GeoJSON: {unmatched}")
    if not rep.empty:
        print(rep["change"].value_counts().to_string())
        if (ow := rep.query("change == 'overwritten'")).shape[0]:
            print("\n⚠️  sheet OVERWROTE existing GeoJSON values:")
            print(ow[["site", "parameter", "old", "new"]].to_string(index=False))
    print("=" * 62)
    return rep


# def _sheet_names_unmatched(matched, key):
#     return []   # populated by caller when the full sheet index is available

### Read the CSDA/Eval Sites GeoJSON stored on GitHub

+ This GeoJSON is built directly off the CSDA Evaluation Sites Database.  
+ The notebook to process this GeoJSON is here: https://github.com/pahbs/csda_summaries/blob/master/notebooks/csda_eval_sites_process.ipynb

In [6]:
TYPE = 'eval' #'csda'

In [7]:
RAW_BASE = 'https://raw.githubusercontent.com/pahbs/csda_summaries/master'
sites_url = f'{RAW_BASE}/sites/{TYPE}_sites_aoi.geojson'
sites = gpd.read_file(sites_url)

/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3

In [8]:
# Get today's date
DATE = datetime.now().strftime('%Y%m%d')
DATE

'20261002'

### Check some useful site attributes

In [9]:
print(list(sites['Evaluation Category'].unique()))

['Geometric', 'Radiometric', 'Radiometric & Geometric', 'InSAR', 'Geometric (vert/horiz)', 'All']


### Indicate a name for the vendor

In [10]:
# Central config: one entry per (vendor, sheet-tab) combination
VENDOR_SHEETS = {
    'Airbus':          ('16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g', 803290261),
    'Vantor_Legion':   ('16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g', 226856298),
    'HydroSat_Kerry':  ('16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g', 0),
    'HydroSat_OR2':    ('16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g', 918595749),
    'OroraTech_OR2':   ('16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g', 1719841802),
    'Tanager':         ('1lkR7cnsqq1EDISoca8tpwcdtfPvWNerbJ5EzCgq-9UI', 0),
    'satellogic_v2':   ('1lkR7cnsqq1EDISoca8tpwcdtfPvWNerbJ5EzCgq-9UI', 0),
    'ESA_misc':        ('16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g', 564318517),
}

In [11]:
# --- Usage ---
VENDOR_NAME = 'HydroSat_Kerry'

In [12]:
sheet_id, gid = VENDOR_SHEETS[VENDOR_NAME]
sheet_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/edit?usp=sharing"

params = load_site_params(sheet_url, gid=gid)
display(params)                      # ← NaN cells are visibly "no opinion"

sites_updated, changes = apply_site_params(sites, params)

#display(changes.query("change != 'unchanged'"))

SITES_FOR_REQUEST = list(params.index)

# which sheet sites never matched the GeoJSON
missing = set(map(_norm, params.index)) - set(map(_norm, sites["Site Name"]))
print("Sheet sites absent from GeoJSON:", missing or "none")

Sheet: 5 sites, 11 parameters resolved
  ℹ️  sheet columns ignored: ['Source']
  cell fill rate:
country                  100 %
    remote_sensing_domain    100 %
    evaluation_category      100 %
    aoi_shape                100 %
    aoi_size_km              100 %
    max_cloud_pct_aoi        100 %
    max_cloud_pct_scene      100 %
    max_view_angle           100 %
    min_num_acqs             100 %
    ideal_num_acqs           100 %
    products                 100 %


,country,remote_sensing_domain,evaluation_category,aoi_shape,aoi_size_km,max_cloud_pct_aoi,max_cloud_pct_scene,max_view_angle,min_num_acqs,ideal_num_acqs,products
site_name,,,,,,,,,,,
Gobabeb TIR,Namibia,Thermal IR,Radiometric,circle,3,0,5,30,1,1,"Level 1B Combined VNIR + LWIR (Day), Level 1B ..."
Lake Tahoe,USA,Thermal IR,Radiometric,circle,3,0,5,30,1,1,"Level 1B Combined VNIR + LWIR (Day), Level 1B ..."
Oklahoma Ag Station,USA,Thermal IR,Radiometric,circle,3,0,5,30,1,1,"Level 1B Combined VNIR + LWIR (Day), Level 1B ..."
Salton Sea,USA,Thermal IR,Radiometric,circle,3,0,5,30,1,1,"Level 1B Combined VNIR + LWIR (Day), Level 1B ..."
Algodones Dunes,USA,Thermal IR,Radiometric,circle,3,0,5,30,1,1,"Level 1B Combined VNIR + LWIR (Day), Level 1B ..."




5 of 617 GeoJSON sites matched the sheet
unchanged      958
filled          45
overwritten      3

⚠️  sheet OVERWROTE existing GeoJSON values:
       site         parameter   old  new
Gobabeb TIR max_cloud_pct_aoi   5.0  0.0
Gobabeb TIR      min_num_acqs   5.0  1.0
Gobabeb TIR    ideal_num_acqs  10.0  1.0
Sheet sites absent from GeoJSON: none


In [13]:
DROP_ON_EXPORT = ["aoi_size_km"]      # confusing; area_km2 conveys it unambiguously

request_gdf = (sites_updated
               .loc[sites_updated["Site Name"].map(lambda s: _norm(s) in
                    {_norm(x) for x in params.index})]
               .drop(columns=DROP_ON_EXPORT, errors="ignore")
               .copy())

request_gdf["area_km2"] = request_gdf["area_km2"].round(3)

cols = ["Site Name", "country", "remote_sensing_domain", "evaluation_category",
        "aoi_shape", "area_km2", "max_cloud_pct_aoi", "max_cloud_pct_scene",
        "max_view_angle", "min_num_acqs", "ideal_num_acqs", "products", "geometry"]

request_gdf = request_gdf[[c for c in cols if c in request_gdf.columns]]


display(request_gdf.drop(columns="geometry"))

,Site Name,country,remote_sensing_domain,evaluation_category,aoi_shape,area_km2,max_cloud_pct_aoi,max_cloud_pct_scene,max_view_angle,min_num_acqs,ideal_num_acqs,products
32,Gobabeb TIR,Namibia,Thermal IR,Radiometric,circle,28.251,0.0,5.0,30.0,1.0,1.0,"Level 1B Combined VNIR + LWIR (Day), Level 1B ..."
69,Lake Tahoe,USA,Thermal IR,Radiometric,circle,28.205,0.0,5.0,30.0,1.0,1.0,"Level 1B Combined VNIR + LWIR (Day), Level 1B ..."
71,Oklahoma Ag Station,USA,Thermal IR,Radiometric,circle,28.246,0.0,5.0,30.0,1.0,1.0,"Level 1B Combined VNIR + LWIR (Day), Level 1B ..."
74,Salton Sea,USA,Thermal IR,Radiometric,circle,28.243,0.0,5.0,30.0,1.0,1.0,"Level 1B Combined VNIR + LWIR (Day), Level 1B ..."
75,Algodones Dunes,USA,Thermal IR,Radiometric,circle,28.229,0.0,5.0,30.0,1.0,1.0,"Level 1B Combined VNIR + LWIR (Day), Level 1B ..."


In [14]:
OUTPUT_DIR = '/home/pmontesa/code/csda_summaries/sites' # Specify your output dir here
out = f'{OUTPUT_DIR}/{TYPE}_sites_aoi_{VENDOR_NAME}_{DATE}.geojson'
request_gdf.to_file(out, driver="GeoJSON")
print(f"{len(request_gdf)} sites → {out}")

5 sites → /home/pmontesa/code/csda_summaries/sites/eval_sites_aoi_HydroSat_Kerry_20261002.geojson


### For Hydrosat requests, need indiv AOI geojsons

In [17]:
import re
from pathlib import Path

OUTPUT_DIR = Path('/home/pmontesa/code/csda_summaries/sites')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def safe_name(s):
    """'PICS Libya-4' -> 'PICS-Libya-4' ; collapse runs, strip edge dashes."""
    s = re.sub(r'\s+', '-', str(s).strip())
    s = re.sub(r'[^A-Za-z0-9._-]', '-', s)
    return re.sub(r'-{2,}', '-', s).strip('-')

written = []
for site, sub in request_gdf.groupby('Site Name', dropna=False):
    aoi = safe_name(site) if pd.notna(site) else 'UNKNOWN'
    out = OUTPUT_DIR / f'{TYPE}_sites_aoi_{VENDOR_NAME}_{aoi}_{DATE}.geojson'
    sub.to_file(out, driver='GeoJSON')
    written.append({'site': site, 'aoi': aoi, 'n_features': len(sub),
                    'file': out.name})
    print(f'  {len(sub):>2} feature(s) → {out.name}')

print(f'\n{len(written)} files written to {OUTPUT_DIR}')
display(pd.DataFrame(written))

   1 feature(s) → eval_sites_aoi_HydroSat_Kerry_Algodones-Dunes_20261002.geojson
   1 feature(s) → eval_sites_aoi_HydroSat_Kerry_Gobabeb-TIR_20261002.geojson
   1 feature(s) → eval_sites_aoi_HydroSat_Kerry_Lake-Tahoe_20261002.geojson
   1 feature(s) → eval_sites_aoi_HydroSat_Kerry_Oklahoma-Ag-Station_20261002.geojson
   1 feature(s) → eval_sites_aoi_HydroSat_Kerry_Salton-Sea_20261002.geojson

5 files written to /home/pmontesa/code/csda_summaries/sites


,site,aoi,n_features,file
0,Algodones Dunes,Algodones-Dunes,1,eval_sites_aoi_HydroSat_Kerry_Algodones-Dunes_...
1,Gobabeb TIR,Gobabeb-TIR,1,eval_sites_aoi_HydroSat_Kerry_Gobabeb-TIR_2026...
2,Lake Tahoe,Lake-Tahoe,1,eval_sites_aoi_HydroSat_Kerry_Lake-Tahoe_20261...
3,Oklahoma Ag Station,Oklahoma-Ag-Station,1,eval_sites_aoi_HydroSat_Kerry_Oklahoma-Ag-Stat...
4,Salton Sea,Salton-Sea,1,eval_sites_aoi_HydroSat_Kerry_Salton-Sea_20261...
